# Advice-marker extended higher-budget study

This notebook carries the complete successor workflow while keeping every mutation, network action, model load, and human-data step disabled by default. The uploaded bundle is verified member by member before source imports. A candidate cap is never inferred: both caps remain unset until a separate completed probe and budget decision exist.

In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

EXPECTED_BUNDLE_SHA256 = "d148bdb43c6545c0a02c5cb75e6b2683f0b56d3dfa31c2e0c25050b3031ed4cb"
PINNED_EMBEDDING_MODEL_ID = "intfloat/e5-small-v2"
PINNED_EMBEDDING_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
PINNED_PREFIX_TOKENIZER_MODEL_ID = "allenai/OLMo-2-0425-1B-Instruct"
PINNED_PREFIX_TOKENIZER_REVISION = "48d788eca847d4d7548f375ad03d3c9312f6139e"
LOCAL_VALIDATION = os.environ.get("EXTENDED_NOTEBOOK_LOCAL_VALIDATION") == "1"

def bootstrap_fs_path(path):
    value = os.fspath(path)
    if os.name != "nt":
        return Path(value)
    if value.startswith("\\\\?\\"):
        return Path(value)
    absolute = os.path.abspath(value)
    return Path("\\\\?\\UNC\\" + absolute[2:] if absolute.startswith("\\\\") else "\\\\?\\" + absolute)

PRIMARY_CAP = None
SENSITIVITY_CAP = None
USE_GOOGLE_DRIVE = False
INSTALL_DEPENDENCIES = False
DOWNLOAD_DETECTORS = False
FREEZE_SUCCESSOR_PROTOCOL = False
CREATE_HUMAN_REVIEW_FORM = False
FREEZE_CALIBRATION_MANIFEST = False
RUN_CALIBRATION = False
EXPORT_CALIBRATION = False
FREEZE_VALIDATION_PLAN = False
RUN_PRIMARY = False
RUN_SENSITIVITY = False
EXPORT_PRIMARY = False
EXPORT_SENSITIVITY = False
RUN_PRIMARY_PREDICTIONS = False
RUN_SENSITIVITY_PREDICTIONS = False
RUN_PRIMARY_ANALYSIS = False
RUN_SENSITIVITY_ANALYSIS = False
PREPARE_VALIDATION_PACKET = False
INIT_HUMAN_PASS_1 = False
INIT_HUMAN_PASS_2 = False
RUN_HUMAN_SCORING = False
DEVICE = "auto"
TORCH_THREADS = 4
ANNOTATOR_1_ID = None
ANNOTATOR_2_ID = None

ACTION_CONTROLS = {
    "use_google_drive": USE_GOOGLE_DRIVE, "install_dependencies": INSTALL_DEPENDENCIES,
    "download_detectors": DOWNLOAD_DETECTORS, "freeze_successor_protocol": FREEZE_SUCCESSOR_PROTOCOL,
    "create_human_review_form": CREATE_HUMAN_REVIEW_FORM, "freeze_calibration_manifest": FREEZE_CALIBRATION_MANIFEST,
    "run_calibration": RUN_CALIBRATION, "export_calibration": EXPORT_CALIBRATION,
    "freeze_validation_plan": FREEZE_VALIDATION_PLAN, "run_primary": RUN_PRIMARY,
    "run_sensitivity": RUN_SENSITIVITY, "export_primary": EXPORT_PRIMARY,
    "export_sensitivity": EXPORT_SENSITIVITY, "run_primary_predictions": RUN_PRIMARY_PREDICTIONS,
    "run_sensitivity_predictions": RUN_SENSITIVITY_PREDICTIONS, "run_primary_analysis": RUN_PRIMARY_ANALYSIS,
    "run_sensitivity_analysis": RUN_SENSITIVITY_ANALYSIS, "prepare_validation_packet": PREPARE_VALIDATION_PACKET,
    "init_human_pass_1": INIT_HUMAN_PASS_1, "init_human_pass_2": INIT_HUMAN_PASS_2,
    "run_human_scoring": RUN_HUMAN_SCORING,
}
print({"local_validation": LOCAL_VALIDATION, "primary_cap": PRIMARY_CAP, "sensitivity_cap": SENSITIVITY_CAP, "all_action_controls_disabled": not any(ACTION_CONTROLS.values()), "action_controls": ACTION_CONTROLS})

{'local_validation': True, 'primary_cap': None, 'sensitivity_cap': None, 'all_action_controls_disabled': True, 'action_controls': {'use_google_drive': False, 'install_dependencies': False, 'download_detectors': False, 'freeze_successor_protocol': False, 'create_human_review_form': False, 'freeze_calibration_manifest': False, 'run_calibration': False, 'export_calibration': False, 'freeze_validation_plan': False, 'run_primary': False, 'run_sensitivity': False, 'export_primary': False, 'export_sensitivity': False, 'run_primary_predictions': False, 'run_sensitivity_predictions': False, 'run_primary_analysis': False, 'run_sensitivity_analysis': False, 'prepare_validation_packet': False, 'init_human_pass_1': False, 'init_human_pass_2': False, 'run_human_scoring': False}}


## Verify and open the source bundle

Outside local validation, upload exactly this bundle. Package installation and Drive mounting require their own switches.

In [2]:
if INSTALL_DEPENDENCIES:
    if LOCAL_VALIDATION:
        raise ValueError("Dependency installation is disabled during local validation.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.57,<5", "huggingface_hub>=0.34,<2", "sentence-transformers>=5,<6", "accelerate>=1.10,<2", "safetensors>=0.6,<1", "numpy>=1.26,<3", "matplotlib>=3.8,<4"], check=True)
else:
    print("Dependency installation skipped.")

Dependency installation skipped.


In [3]:
bundle_env = os.environ.get("EXTENDED_NOTEBOOK_BUNDLE")
if bundle_env:
    bundle_path = bootstrap_fs_path(bundle_env)
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Set EXTENDED_NOTEBOOK_BUNDLE outside Colab.") from exc
    uploaded = files.upload()
    candidates = [Path(name) for name in uploaded if name.endswith(".zip")]
    if len(candidates) != 1:
        raise ValueError("Upload exactly one extended-study bundle.")
    bundle_path = bootstrap_fs_path(candidates[0])
if hashlib.sha256(bundle_path.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA256:
    raise ValueError("Bundle SHA-256 mismatch.")
with zipfile.ZipFile(bundle_path) as archive:
    manifest = json.loads(archive.read("bundle_manifest.json"))
    for info in archive.infolist():
        member = Path(info.filename)
        if member.is_absolute() or ".." in member.parts:
            raise ValueError(f"Unsafe archive member: {info.filename}")
    workspace_parent = bootstrap_fs_path(os.environ.get("EXTENDED_NOTEBOOK_WORKSPACE", "/tmp/advice_marker_extended_bundles"))
    workspace = workspace_parent / manifest["bundle_content_id"]
    workspace.mkdir(parents=True, exist_ok=True)
    archive.extractall(workspace)
for relative, expected in manifest["files"].items():
    target = workspace / relative
    if hashlib.sha256(target.read_bytes()).hexdigest() != expected["sha256"] or target.stat().st_size != expected["size_bytes"]:
        raise ValueError(f"Bundled file mismatch: {relative}")
print({"bundle_content_id": manifest["bundle_content_id"], "verified_files": len(manifest["files"]), "baseline_files": manifest["verified_baseline"]["files_verified"], "probe_input_status": manifest["probe_input"]["status"]})

{'bundle_content_id': 'extended_bundle_6550d81c61f8ffc57b4c12e9773248dc', 'verified_files': 37, 'baseline_files': 25, 'probe_input_status': 'pending_complete_probe_run_export_and_budget_decision'}


In [4]:
if USE_GOOGLE_DRIVE:
    if LOCAL_VALIDATION:
        raise ValueError("Drive mounting is disabled during local validation.")
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_ROOT = Path("/content/drive/MyDrive/advice_marker_extended_study")
else:
    CACHE_ROOT = Path(os.environ.get("EXTENDED_NOTEBOOK_CACHE", "/tmp/advice_marker_extended_cache"))
sys.path.insert(0, str(workspace))
from experiments import extended_revision_study as core
from experiments import extended_calibration_study as calibration
from experiments import extended_prospective_validation as prospective
from experiments import extended_sensitivity_analysis as sensitivity_analysis
from experiments import development_budget_probe as budget_probe
from experiments import revision_analysis, validation_v4

CACHE_ROOT = core.fs_path(CACHE_ROOT)
workspace = core.fs_path(workspace)
parent_protocol_path = core.fs_path(workspace / "output/revision/full_study/protocol_v2.json")
fixed_probe_manifest_path = core.fs_path(workspace / "output/revision/development_budget_probe/development_1024_probe_manifest_v1.json")
annotation_guide_path = core.fs_path(workspace / "out/paper/ANNOTATION_GUIDE.md")
parent_protocol = core._parent_protocol(parent_protocol_path)
fixed_probe_manifest = budget_probe.load_manifest(fixed_probe_manifest_path, parent_protocol_path)
print({"imports": [core.__name__, calibration.__name__, prospective.__name__, sensitivity_analysis.__name__], "parent_protocol_id": parent_protocol["protocol_id"], "fixed_probe_manifest_id": fixed_probe_manifest["probe_manifest_id"], "fixed_probe_expected": fixed_probe_manifest["expected_responses"]})

{'imports': ['experiments.extended_revision_study', 'experiments.extended_calibration_study', 'experiments.extended_prospective_validation', 'experiments.extended_sensitivity_analysis'], 'parent_protocol_id': 'protocol_32f4c8538587e45cdf07e88ed9b1f77d', 'fixed_probe_manifest_id': 'budget_probe_manifest_7d029d7622f863fb50f1a468042d4700', 'fixed_probe_expected': 8}


## Completed probe and explicit budget decision

The source bundle contains the fixed manifest and preceding 256/512 evidence. The separately verified completed-probe companion supplies the 8/8 run and exact export under their repository-relative paths. A budget decision and both caps remain unset and must be supplied separately before freezing.

In [5]:
PROBE_RUN_DIR = core.fs_path(os.environ["EXTENDED_PROBE_RUN_DIR"]) if os.environ.get("EXTENDED_PROBE_RUN_DIR") else None
PROBE_EXPORT_PATH = core.fs_path(os.environ["EXTENDED_PROBE_EXPORT"]) if os.environ.get("EXTENDED_PROBE_EXPORT") else None
BUDGET_DECISION_PATH = core.fs_path(os.environ["EXTENDED_BUDGET_DECISION"]) if os.environ.get("EXTENDED_BUDGET_DECISION") else None
SUCCESSOR_PROTOCOL_PATH = core.fs_path(os.environ.get("EXTENDED_PROTOCOL", str(CACHE_ROOT / "full_study/protocol_v1.json")))
protocol = core.load_protocol(SUCCESSOR_PROTOCOL_PATH) if SUCCESSOR_PROTOCOL_PATH.is_file() else None
if PRIMARY_CAP is None or SENSITIVITY_CAP is None:
    print("Pending: set both explicit caps only after the completed probe and decision are available.")
else:
    candidate = core.build_protocol(parent_protocol_path, PRIMARY_CAP, SENSITIVITY_CAP)
    print({"candidate_protocol_id": candidate["candidate_protocol_id"], "primary": len(candidate["primary_grid"]), "sensitivity": len(candidate["longer_budget_sensitivity"]["grid"])})
if FREEZE_SUCCESSOR_PROTOCOL:
    if None in (PRIMARY_CAP, SENSITIVITY_CAP, PROBE_RUN_DIR, PROBE_EXPORT_PATH, BUDGET_DECISION_PATH):
        raise ValueError("Freezing requires explicit caps, complete probe run/export, and budget decision.")
    protocol = core.freeze_protocol(parent_protocol_path, PRIMARY_CAP, SENSITIVITY_CAP, SUCCESSOR_PROTOCOL_PATH, probe_manifest_path=fixed_probe_manifest_path, probe_run_dir=PROBE_RUN_DIR, probe_export_path=PROBE_EXPORT_PATH, budget_decision_path=BUDGET_DECISION_PATH)
    print({"frozen_protocol_id": protocol["protocol_id"]})
else:
    print("Successor protocol freezing disabled.")

Pending: set both explicit caps only after the completed probe and decision are available.
Successor protocol freezing disabled.


## Genuine prompt review and 96-response calibration

In [6]:
HUMAN_REVIEW_PATH = core.fs_path(os.environ.get("EXTENDED_HUMAN_REVIEW", str(CACHE_ROOT / "human_prompt_review.json")))
if CREATE_HUMAN_REVIEW_FORM:
    if protocol is None:
        raise ValueError("Freeze the successor protocol before creating its blank review form.")
    core.create_human_review_manifest(protocol, HUMAN_REVIEW_PATH)
    print({"blank_review_form": str(HUMAN_REVIEW_PATH), "human_fields_completed": 0})
elif protocol is not None and HUMAN_REVIEW_PATH.is_file():
    core.validate_human_review(protocol, HUMAN_REVIEW_PATH)
    print("Genuine completed prompt review validated.")
else:
    print("Prompt review pending; no form or judgments were created.")

Prompt review pending; no form or judgments were created.


In [7]:
CALIBRATION_ROOT = core.fs_path(CACHE_ROOT / "calibration_study")
CALIBRATION_MANIFEST_PATH = core.fs_path(os.environ.get("EXTENDED_CALIBRATION_MANIFEST", str(CALIBRATION_ROOT / "manifest.json")))
CALIBRATION_RUN_DIR = core.fs_path(os.environ["EXTENDED_CALIBRATION_RUN_DIR"]) if os.environ.get("EXTENDED_CALIBRATION_RUN_DIR") else None
CALIBRATION_RESPONSES_PATH = core.fs_path(os.environ.get("EXTENDED_CALIBRATION_RESPONSES", str(CALIBRATION_ROOT / "responses.jsonl")))
calibration_result = None
if FREEZE_CALIBRATION_MANIFEST:
    if protocol is None or PRIMARY_CAP is None:
        raise ValueError("Frozen successor protocol and explicit primary cap required.")
    calibration.freeze_manifest(SUCCESSOR_PROTOCOL_PATH, PRIMARY_CAP, CALIBRATION_MANIFEST_PATH)
if RUN_CALIBRATION:
    calibration_result = calibration.run_generation(CALIBRATION_MANIFEST_PATH, SUCCESSOR_PROTOCOL_PATH, CALIBRATION_ROOT, DEVICE, False, TORCH_THREADS)
    CALIBRATION_RUN_DIR = core.fs_path(calibration_result["run_dir"])
if EXPORT_CALIBRATION:
    calibration.export_run(CALIBRATION_MANIFEST_PATH, SUCCESSOR_PROTOCOL_PATH, CALIBRATION_RUN_DIR, CALIBRATION_RESPONSES_PATH)
if protocol is not None and CALIBRATION_MANIFEST_PATH.is_file() and CALIBRATION_RUN_DIR is not None and CALIBRATION_RESPONSES_PATH.is_file():
    report = calibration.verify_run(CALIBRATION_MANIFEST_PATH, SUCCESSOR_PROTOCOL_PATH, CALIBRATION_RUN_DIR, require_complete=True)
    print({"calibration": report, "responses_export": str(CALIBRATION_RESPONSES_PATH)})
else:
    print("Calibration pending: requires an exact frozen 96-response manifest, complete run, and export.")

Calibration pending: requires an exact frozen 96-response manifest, complete run, and export.


## Human-reviewed development evidence and prospective plan

At least 60 distinct calibration responses must be genuinely reviewed, with a nonempty timed subset recorded in the evidence file. The validator enforces these requirements; this notebook never fills human fields.

In [8]:
DEVELOPMENT_EVIDENCE_PATH = core.fs_path(os.environ.get("EXTENDED_DEVELOPMENT_EVIDENCE", str(CALIBRATION_ROOT / "human_development_evidence.json")))
VALIDATION_PLAN_PATH = core.fs_path(os.environ.get("EXTENDED_VALIDATION_PLAN", str(CACHE_ROOT / "prospective_validation/sampling_plan_v1.json")))
if FREEZE_VALIDATION_PLAN:
    prospective.freeze_plan(SUCCESSOR_PROTOCOL_PATH, CALIBRATION_MANIFEST_PATH, CALIBRATION_RESPONSES_PATH, CALIBRATION_RUN_DIR, DEVELOPMENT_EVIDENCE_PATH, VALIDATION_PLAN_PATH, annotation_guide_path)
if protocol is not None and all(path is not None and core.fs_path(path).exists() for path in (CALIBRATION_MANIFEST_PATH, CALIBRATION_RESPONSES_PATH, CALIBRATION_RUN_DIR, DEVELOPMENT_EVIDENCE_PATH)):
    if VALIDATION_PLAN_PATH.is_file():
        print(prospective.validate_plan(VALIDATION_PLAN_PATH, SUCCESSOR_PROTOCOL_PATH, CALIBRATION_MANIFEST_PATH, CALIBRATION_RESPONSES_PATH, CALIBRATION_RUN_DIR, DEVELOPMENT_EVIDENCE_PATH, annotation_guide_path))
    else:
        print(prospective.preview_plan(SUCCESSOR_PROTOCOL_PATH, CALIBRATION_MANIFEST_PATH, CALIBRATION_RESPONSES_PATH, CALIBRATION_RUN_DIR, annotation_guide_path))
else:
    print("Prospective plan pending complete calibration and genuine reviewed/timed development evidence.")

def readiness_mapping():
    return {"plan_path": VALIDATION_PLAN_PATH, "calibration_manifest_path": CALIBRATION_MANIFEST_PATH, "calibration_responses_path": CALIBRATION_RESPONSES_PATH, "calibration_run_dir": CALIBRATION_RUN_DIR, "evidence_path": DEVELOPMENT_EVIDENCE_PATH, "guide_path": annotation_guide_path}

Prospective plan pending complete calibration and genuine reviewed/timed development evidence.


## Complete primary and longer-budget generation

Both runs use strict readiness gates and safe per-response resume. Primary is always 2,880 responses; sensitivity is always the pre-specified 192 responses.

In [9]:
PRIMARY_RUN_DIR = core.fs_path(os.environ["EXTENDED_PRIMARY_RUN_DIR"]) if os.environ.get("EXTENDED_PRIMARY_RUN_DIR") else None
SENSITIVITY_RUN_DIR = core.fs_path(os.environ["EXTENDED_SENSITIVITY_RUN_DIR"]) if os.environ.get("EXTENDED_SENSITIVITY_RUN_DIR") else None
primary_result = sensitivity_result = None
if RUN_PRIMARY:
    primary_result = core.run_generation(SUCCESSOR_PROTOCOL_PATH, "evaluation", CACHE_ROOT, None, HUMAN_REVIEW_PATH, DEVICE, False, TORCH_THREADS, readiness=readiness_mapping())
    PRIMARY_RUN_DIR = core.fs_path(primary_result["run_dir"])
if RUN_SENSITIVITY:
    sensitivity_result = core.run_generation(SUCCESSOR_PROTOCOL_PATH, "sensitivity", CACHE_ROOT, None, HUMAN_REVIEW_PATH, DEVICE, False, TORCH_THREADS, readiness=readiness_mapping())
    SENSITIVITY_RUN_DIR = core.fs_path(sensitivity_result["run_dir"])
print({"primary": primary_result or "disabled", "sensitivity": sensitivity_result or "disabled"})

{'primary': 'disabled', 'sensitivity': 'disabled'}


In [10]:
PRIMARY_RESPONSES_PATH = core.fs_path(os.environ.get("EXTENDED_PRIMARY_RESPONSES", str(CACHE_ROOT / "exports/primary_responses.jsonl")))
SENSITIVITY_RESPONSES_PATH = core.fs_path(os.environ.get("EXTENDED_SENSITIVITY_RESPONSES", str(CACHE_ROOT / "exports/sensitivity_responses.jsonl")))
if EXPORT_PRIMARY:
    core.export_run(SUCCESSOR_PROTOCOL_PATH, PRIMARY_RUN_DIR, PRIMARY_RESPONSES_PATH, require_complete=True)
if EXPORT_SENSITIVITY:
    core.export_run(SUCCESSOR_PROTOCOL_PATH, SENSITIVITY_RUN_DIR, SENSITIVITY_RESPONSES_PATH, require_complete=True)
print({"primary_export_enabled": EXPORT_PRIMARY, "sensitivity_export_enabled": EXPORT_SENSITIVITY})

{'primary_export_enabled': False, 'sensitivity_export_enabled': False}


## 8,640 primary and 576 sensitivity detector rows

In [11]:
if DOWNLOAD_DETECTORS:
    if LOCAL_VALIDATION:
        raise ValueError("Detector downloads are disabled during local validation.")
    from huggingface_hub import snapshot_download
    snapshot_download(PINNED_EMBEDDING_MODEL_ID, revision=PINNED_EMBEDDING_REVISION)
    snapshot_download(PINNED_PREFIX_TOKENIZER_MODEL_ID, revision=PINNED_PREFIX_TOKENIZER_REVISION)
PRIMARY_PREDICTIONS_PATH = core.fs_path(os.environ.get("EXTENDED_PRIMARY_PREDICTIONS", str(CACHE_ROOT / "exports/primary_predictions.jsonl")))
SENSITIVITY_PREDICTIONS_PATH = core.fs_path(os.environ.get("EXTENDED_SENSITIVITY_PREDICTIONS", str(CACHE_ROOT / "exports/sensitivity_predictions.jsonl")))
if RUN_PRIMARY_PREDICTIONS:
    report = core.predict_complete_evaluation(SUCCESSOR_PROTOCOL_PATH, PRIMARY_RUN_DIR, PRIMARY_PREDICTIONS_PATH)
    if report["prediction_rows"] != 8640: raise ValueError("Primary prediction count mismatch")
if RUN_SENSITIVITY_PREDICTIONS:
    report = sensitivity_analysis.predict_complete_sensitivity(SUCCESSOR_PROTOCOL_PATH, SENSITIVITY_RUN_DIR, SENSITIVITY_PREDICTIONS_PATH)
    if report["prediction_rows"] != 576: raise ValueError("Sensitivity prediction count mismatch")
print({"primary_prediction_action": RUN_PRIMARY_PREDICTIONS, "sensitivity_prediction_action": RUN_SENSITIVITY_PREDICTIONS})

{'primary_prediction_action': False, 'sensitivity_prediction_action': False}


## Primary analysis and matched seed-zero sensitivity

In [12]:
PRIMARY_ANALYSIS_DIR = core.fs_path(CACHE_ROOT / "primary_analysis")
SENSITIVITY_ANALYSIS_DIR = core.fs_path(CACHE_ROOT / "sensitivity_analysis")
if RUN_PRIMARY_ANALYSIS:
    subprocess.run([sys.executable, "-m", "experiments.revision_analysis", "--protocol", str(SUCCESSOR_PROTOCOL_PATH), "--responses", str(PRIMARY_RESPONSES_PATH), "--predictions", str(PRIMARY_PREDICTIONS_PATH), "--output-dir", str(PRIMARY_ANALYSIS_DIR), "--bootstrap-replicates", "10000", "--bootstrap-seed", "20260915", "--confirm-real-complete-archive"], check=True, cwd=workspace)
if RUN_SENSITIVITY_ANALYSIS:
    sensitivity_analysis.analyze(SUCCESSOR_PROTOCOL_PATH, PRIMARY_RUN_DIR, PRIMARY_RESPONSES_PATH, PRIMARY_PREDICTIONS_PATH, SENSITIVITY_RUN_DIR, SENSITIVITY_RESPONSES_PATH, SENSITIVITY_PREDICTIONS_PATH, SENSITIVITY_ANALYSIS_DIR, bootstrap_replicates=10000, seed=20260915)
print({"primary_analysis": RUN_PRIMARY_ANALYSIS, "matched_seed0_sensitivity": RUN_SENSITIVITY_ANALYSIS})

{'primary_analysis': False, 'matched_seed0_sensitivity': False}


## Prospective packet, two human passes, and scoring

In [13]:
VALIDATION_PACKET_DIR = core.fs_path(os.environ.get("EXTENDED_VALIDATION_PACKET", str(CACHE_ROOT / "annotation_v4/evaluation_packet")))
if PREPARE_VALIDATION_PACKET:
    prospective.prepare_packet(VALIDATION_PLAN_PATH, SUCCESSOR_PROTOCOL_PATH, PRIMARY_RESPONSES_PATH, PRIMARY_PREDICTIONS_PATH, CALIBRATION_MANIFEST_PATH, CALIBRATION_RESPONSES_PATH, CALIBRATION_RUN_DIR, DEVELOPMENT_EVIDENCE_PATH, PRIMARY_RUN_DIR, VALIDATION_PACKET_DIR, annotation_guide_path)
if INIT_HUMAN_PASS_1:
    if not ANNOTATOR_1_ID: raise ValueError("A real annotator ID is required for pass 1.")
    validation_v4.init_annotation(1, ANNOTATOR_1_ID, True, VALIDATION_PACKET_DIR)
if INIT_HUMAN_PASS_2:
    if not ANNOTATOR_2_ID: raise ValueError("A real annotator ID is required for pass 2.")
    validation_v4.init_annotation(2, ANNOTATOR_2_ID, True, VALIDATION_PACKET_DIR)
if RUN_HUMAN_SCORING:
    print(validation_v4.score(VALIDATION_PACKET_DIR, 10000, 20260915))
else:
    print({"packet_preparation": PREPARE_VALIDATION_PACKET, "human_pass_1": INIT_HUMAN_PASS_1, "human_pass_2": INIT_HUMAN_PASS_2, "human_scoring": False})

{'packet_preparation': False, 'human_pass_1': False, 'human_pass_2': False, 'human_scoring': False}


## Status

The default execution proves only source/bundle integrity and workflow importability. Scientific completion requires a completed 8/8 probe, explicit budget decision, frozen successor protocol, genuine prompt review, complete 96-response calibration plus reviewed timing evidence, frozen prospective plan, 2,880 primary and 192 sensitivity responses, 8,640 and 576 prediction rows, and two genuine human annotation passes.